# 03 — Feature Engineering

Project: E-Waste Profitability Prediction
Group: 2026-AI-14
Module: IT3091 — Machine Learning

Objectives:
- Create metal-to-weight ratios.
- Create a logarithmic weight feature.
- Define a training-derived profit-tier policy.
- Document feature definitions and leakage risks.
- Export engineered training and test datasets.

Regression target: total_profit_usd
Classification target: Batch_Priority

Batch_Priority is a profit-tier proxy, not independently
verified operational scheduling ground truth.

Libraries import

In [ ]:
import pandas as pd
import numpy as np
import json

from google.colab import files

pd.set_option("display.max_columns", None)

Cleaned datasets upload

In [ ]:
uploaded = files.upload()

Saving train_clean.csv to train_clean.csv
Saving test_clean.csv to test_clean.csv


In [ ]:
required_files = [
    "train_clean.csv",
    "test_clean.csv"
]

for filename in required_files:
    assert filename in uploaded, f"Missing file: {filename}"

train_clean = pd.read_csv("train_clean.csv")
test_clean = pd.read_csv("test_clean.csv")

print("Training shape:", train_clean.shape)
print("Test shape:", test_clean.shape)

display(train_clean.head())

Training shape: (16000, 13)
Test shape: (4000, 13)


,device_id,item_name,device_category,condition,weight_g,device_age_years,gold_yield_g,silver_yield_g,copper_yield_g,total_profit_usd,Acquisition_Cost_LKR,Hazardous_Flag,Casing_Material
0,DEV-05895,Nikon D3500,Digital Camera,Used-Fair,440.58,7.0,0.0304,1.1019,42.93,2.53,13242.66,Yes,Plastic
1,DEV-03729,iPhone 11,Smartphone,Used-Good,179.41,8.0,0.0261,0.2686,17.22,1.84,11511.64,Yes,Mixed Metal
2,DEV-08959,Samsung Galaxy Watch 4,Smartwatch,New,42.33,3.0,0.0096,0.0749,3.94,0.94,14879.31,Yes,Aluminum
3,DEV-07672,iPhone 13,Smartphone,Used-Fair,178.62,6.0,0.0255,0.3385,17.12,1.46,19217.96,No,Aluminum
4,DEV-06000,Intel Core i3-10100,Microprocessor,E-Waste / End-of-Life,59.42,4.0,0.4903,0.1835,12.50,17.35,1252.62,No,Mixed Metal


Split consistency

In [ ]:
assert train_clean.columns.tolist() == (
    test_clean.columns.tolist()
)

for dataset in [train_clean, test_clean]:
    assert dataset["device_id"].notna().all()
    assert dataset["device_id"].is_unique
    assert dataset["total_profit_usd"].notna().all()

assert set(train_clean["device_id"]).isdisjoint(
    set(test_clean["device_id"])
)

assert len(train_clean) == 16000
assert len(test_clean) == 4000

print("Split and target checks passed.")

Split and target checks passed.


Feature definitions

## Engineered features

1. gold_yield_ratio = gold_yield_g / weight_g
2. silver_yield_ratio = silver_yield_g / weight_g
3. copper_yield_ratio = copper_yield_g / weight_g
4. total_metal_ratio =
   (gold_yield_g + silver_yield_g + copper_yield_g) / weight_g
5. log_weight_g = ln(1 + weight_g)

Yield ratios are gram-per-gram fractions, not percentages.
Multiply by 100 only when displaying them as percentages.

Missing or non-positive weight produces a missing ratio.
Missing measurements remain missing for fold-local imputation.

These calculations use each record's inputs only.
They do not use profit or dataset-wide fitted parameters.

Metal yields must be available before the intended decision.

Reusable feature-engineering function

In [ ]:
def engineer_features(dataset):
    engineered = dataset.copy()

    valid_weight = engineered["weight_g"].where(
        engineered["weight_g"] > 0
    )

    for metal in ["gold", "silver", "copper"]:
        engineered[f"{metal}_yield_ratio"] = (
            engineered[f"{metal}_yield_g"] / valid_weight
        )

    # Require all three measurements; do not treat missing
    # metal yield as zero.
    total_metal_g = engineered[
        [
            "gold_yield_g",
            "silver_yield_g",
            "copper_yield_g"
        ]
    ].sum(axis=1, min_count=3)

    engineered["total_metal_ratio"] = (
        total_metal_g / valid_weight
    )

    engineered["log_weight_g"] = np.log1p(
        valid_weight
    )

    engineered = engineered.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return engineered

Apply both Training  and test data

In [ ]:
train_features = engineer_features(train_clean)
test_features = engineer_features(test_clean)

engineered_columns = [
    "gold_yield_ratio",
    "silver_yield_ratio",
    "copper_yield_ratio",
    "total_metal_ratio",
    "log_weight_g"
]

display(
    train_features[
        ["device_id", "weight_g"] + engineered_columns
    ].head(10)
)

display(
    train_features[engineered_columns].describe().T
)

print("Missing engineered values:")
print(
    train_features[engineered_columns].isna().sum()
)

,device_id,weight_g,gold_yield_ratio,silver_yield_ratio,copper_yield_ratio,total_metal_ratio,log_weight_g
0,DEV-05895,440.58,0.000069,0.002501,0.097440,0.100010,6.090359
1,DEV-03729,179.41,0.000145,0.001497,0.095981,0.097624,5.195232
2,DEV-08959,42.33,0.000227,0.001769,0.093078,0.095074,3.768845
3,DEV-07672,178.62,0.000143,0.001895,0.095846,0.097884,5.190844
4,DEV-06000,59.42,0.008251,0.003088,0.210367,0.221706,4.101320
5,DEV-05752,16.95,0.001764,0.003746,0.096755,0.102265,2.887590
6,DEV-01689,560.56,0.000101,0.001360,0.072820,0.074281,6.330719
7,DEV-06837,165.25,0.000193,0.001744,0.091256,0.093193,5.113493
8,DEV-06537,46.36,0.000142,0.001387,0.067947,0.069476,3.857778
9,DEV-04843,687.13,0.000560,0.003014,0.180126,0.183700,6.533978


,count,mean,std,min,25%,50%,75%,max
gold_yield_ratio,16000.0,0.000955,0.001991,0.000013,0.000102,0.000163,0.001000,0.015818
silver_yield_ratio,16000.0,0.002498,0.001863,0.000413,0.001251,0.001729,0.003350,0.009771
copper_yield_ratio,16000.0,0.131675,0.067363,0.040734,0.087076,0.109630,0.155657,0.486799
total_metal_ratio,16000.0,0.135129,0.069298,0.042089,0.089403,0.112343,0.159478,0.508673
log_weight_g,16000.0,5.343835,1.393862,2.397895,3.991158,5.838139,6.239467,7.824350


Missing engineered values:
gold_yield_ratio      0
silver_yield_ratio    0
copper_yield_ratio    0
total_metal_ratio     0
log_weight_g          0
dtype: int64


Feature calculations verify

In [ ]:
for original, engineered in [
    (train_clean, train_features),
    (test_clean, test_features)
]:
    assert original["device_id"].equals(
        engineered["device_id"]
    )

    assert np.allclose(
        original["total_profit_usd"],
        engineered["total_profit_usd"],
        equal_nan=True
    )

valid_gold_rows = (
    train_features["weight_g"].gt(0) &
    train_features["gold_yield_g"].notna()
)

assert np.allclose(
    train_features.loc[
        valid_gold_rows, "gold_yield_ratio"
    ],
    (
        train_features.loc[
            valid_gold_rows, "gold_yield_g"
        ] /
        train_features.loc[
            valid_gold_rows, "weight_g"
        ]
    )
)

assert not np.isinf(
    train_features[engineered_columns].to_numpy()
).any()

assert not np.isinf(
    test_features[engineered_columns].to_numpy()
).any()

ratio_flags = train_features.loc[
    train_features["total_metal_ratio"] > 1,
    ["device_id", "weight_g"] + engineered_columns
]

print("Feature calculation checks passed.")
print("Training total-metal ratios > 1:", len(ratio_flags))

display(ratio_flags.head())

Feature calculation checks passed.
Training total-metal ratios > 1: 0


,device_id,weight_g,gold_yield_ratio,silver_yield_ratio,copper_yield_ratio,total_metal_ratio,log_weight_g


Get priority cutoffs from Training data

In [ ]:
low_cutoff, high_cutoff = (
    train_features["total_profit_usd"]
    .quantile([1 / 3, 2 / 3])
    .to_numpy()
)

assert np.isfinite(low_cutoff)
assert np.isfinite(high_cutoff)
assert low_cutoff < high_cutoff

print("Low cutoff USD:", round(low_cutoff, 4))
print("High cutoff USD:", round(high_cutoff, 4))



Low cutoff USD: 1.57
High cutoff USD: 4.77


 Make Priority labels

In [ ]:
def assign_profit_tier(profit_values, low, high):
    values = np.asarray(profit_values, dtype=float)

    assert np.isfinite(values).all()

    return np.where(
        values <= low,
        "Low",
        np.where(
            values <= high,
            "Medium",
            "High"
        )
    )

train_features["Batch_Priority"] = assign_profit_tier(
    train_features["total_profit_usd"],
    low_cutoff,
    high_cutoff
)

test_features["Batch_Priority"] = assign_profit_tier(
    test_features["total_profit_usd"],
    low_cutoff,
    high_cutoff
)

display(
    train_features[
        ["device_id", "total_profit_usd", "Batch_Priority"]
    ].head(10)
)

,device_id,total_profit_usd,Batch_Priority
0,DEV-05895,2.53,Medium
1,DEV-03729,1.84,Medium
2,DEV-08959,0.94,Low
3,DEV-07672,1.46,Low
4,DEV-06000,17.35,High
5,DEV-05752,1.07,Low
6,DEV-01689,6.66,High
7,DEV-06837,2.14,Medium
8,DEV-06537,0.48,Low
9,DEV-04843,24.95,High


Training class balance

In [ ]:
priority_order = ["Low", "Medium", "High"]

priority_counts = (
    train_features["Batch_Priority"]
    .value_counts()
    .reindex(priority_order, fill_value=0)
    .rename("count")
    .to_frame()
)

priority_counts["percentage"] = (
    priority_counts["count"] /
    len(train_features) * 100
).round(2)

display(priority_counts)

assert (priority_counts["count"] >= 5).all(), (
    "Each class needs sufficient records for five-fold CV."
)

priority_counts.to_csv(
    "training_priority_counts.csv"
)

,count,percentage
Batch_Priority,,
Low,5339,33.37
Medium,5328,33.30
High,5333,33.33


## Priority policy limitation

The labels are derived from recorded profit using training-only
tertile thresholds. They measure profit-tier prediction.

They do not incorporate processing time, energy, contamination,
capacity or validated operational scheduling decisions.

Classifier cross-validation will be conditional on this policy,
whose thresholds were learned from the full training targets.
It does not independently validate the threshold policy itself.

Stakeholder-approved fixed thresholds or genuine operational
labels are needed to validate an operational priority system.

Make Final model input lists

In [ ]:
numeric_features = [
    "weight_g",
    "device_age_years",
    "gold_yield_g",
    "silver_yield_g",
    "copper_yield_g",
    "Acquisition_Cost_LKR",
    "gold_yield_ratio",
    "silver_yield_ratio",
    "copper_yield_ratio",
    "total_metal_ratio",
    "log_weight_g"
]

categorical_features = [
    "device_category",
    "condition",
    "Hazardous_Flag",
    "Casing_Material"
]

feature_columns = numeric_features + categorical_features

excluded_columns = [
    "device_id",
    "item_name",
    "total_profit_usd",
    "Batch_Priority"
]

assert not set(feature_columns) & set(excluded_columns)

X_train = train_features[feature_columns].copy()
X_test = test_features[feature_columns].copy()

y_train_regression = train_features["total_profit_usd"]
y_test_regression = test_features["total_profit_usd"]

y_train_classification = train_features["Batch_Priority"]
y_test_classification = test_features["Batch_Priority"]

print("Training inputs:", X_train.shape)
print("Test inputs:", X_test.shape)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Training inputs: (16000, 15)
Test inputs: (4000, 15)
Numeric features: 11
Categorical features: 4


Create Feature Engineering Log

In [ ]:
feature_log = pd.DataFrame([
    {
        "feature": "gold_yield_ratio",
        "definition": "gold_yield_g / weight_g",
        "reason": "Gold yield relative to item weight"
    },
    {
        "feature": "silver_yield_ratio",
        "definition": "silver_yield_g / weight_g",
        "reason": "Silver yield relative to item weight"
    },
    {
        "feature": "copper_yield_ratio",
        "definition": "copper_yield_g / weight_g",
        "reason": "Copper yield relative to item weight"
    },
    {
        "feature": "total_metal_ratio",
        "definition": "Sum of three metal yields / weight_g",
        "reason": "Combined metal yield relative to weight"
    },
    {
        "feature": "log_weight_g",
        "definition": "ln(1 + weight_g)",
        "reason": "Candidate representation for skewed weight"
    },
    {
        "feature": "Batch_Priority",
        "definition": "Training-profit tertile policy",
        "reason": "Secondary target proxy; never an input"
    },
    {
        "feature": "Unavailable domain features",
        "definition": "Energy, acid, processing time",
        "reason": "Not created because source columns are absent"
    },
    {
        "feature": "Rejected target-derived inputs",
        "definition": "Actual profit per kg; actual profit tier",
        "reason": "Would leak target information into inputs"
    }
])

display(feature_log)

feature_log.to_csv(
    "feature_engineering_log.csv",
    index=False
)

,feature,definition,reason
0,gold_yield_ratio,gold_yield_g / weight_g,Gold yield relative to item weight
1,silver_yield_ratio,silver_yield_g / weight_g,Silver yield relative to item weight
2,copper_yield_ratio,copper_yield_g / weight_g,Copper yield relative to item weight
3,total_metal_ratio,Sum of three metal yields / weight_g,Combined metal yield relative to weight
4,log_weight_g,ln(1 + weight_g),Candidate representation for skewed weight
5,Batch_Priority,Training-profit tertile policy,Secondary target proxy; never an input
6,Unavailable domain features,"Energy, acid, processing time",Not created because source columns are absent
7,Rejected target-derived inputs,Actual profit per kg; actual profit tier,Would leak target information into inputs


save Policy and datasets

In [ ]:
feature_config = {
    "regression_target": "total_profit_usd",
    "classification_target": "Batch_Priority",
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "feature_columns": feature_columns,
    "raw_feature_columns": [
        column for column in feature_columns
        if column not in engineered_columns
    ],
    "engineered_columns": engineered_columns,
    "excluded_columns": excluded_columns,
    "priority_policy": {
        "low_cutoff_usd": float(low_cutoff),
        "high_cutoff_usd": float(high_cutoff),
        "rule": (
            "Low: profit <= low; "
            "Medium: low < profit <= high; "
            "High: profit > high"
        ),
        "threshold_source": "Training targets only",
        "meaning": "Profit-tier proxy"
    },
    "split_preserved": True
}

with open("feature_config.json", "w") as file:
    json.dump(feature_config, file, indent=2)

train_features.to_csv(
    "train_features.csv",
    index=False
)

test_features.to_csv(
    "test_features.csv",
    index=False
)

print("Engineered datasets and configuration saved.")

Engineered datasets and configuration saved.


download Logs and configuration

In [ ]:
files.download("feature_engineering_log.csv")
files.download("feature_config.json")
files.download("training_priority_counts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("train_features.csv")
files.download("test_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>